# Week 6 Day 4: LangGraph Integration — Routing Between Chat, Retrieval & Prediction

### Goal
Today we'll be connecting everything built so far into one LangGraph app:
- Day 2's prediction models (match winner, top player by stat)
- Day 3's retrieval tools and AFL-only chat agent

The app needs to figure out what kind of question it got (factual, retrieval, prediction, or off-topic) and route it to the right place, instead of relying on one agent to freely decide everything

Reusing as is from before:
- `team_match_features_v1_2026-07-27.csv`, `player_match_features_v1_2026-07-27.csv`
- `models/` folder (match_winner_model, top_player models, latest_team_features, latest_player_features, valid_teams)
- `predict.py` functions and the retrieval tools + system prompt from `afl_chat_agent.py`

## Loading and Setup

In [6]:
pip install scikit-learn

  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/3b/67/be3d369f40d8178ba3bd86635d132e08cb5329b023e4669d9426d84bc007/scikit_learn-1.9.0-cp311-cp311-win_amd64.whl.metadata
  Using cached scikit_learn-1.9.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/95/da/0d1df507cf574b3f224ccc3d45244c9a1d732c81dcb26b1e8a766ae271a8/scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Obtaining dependency information for narwhals>=2.0.1 from https://files.pythonhosted.org/packages/7e/85/a5bfaebfd305ac18b57b0854d74e37e586809061a91fda62f0bd50c8518e/narwhals-2.24.0-py3-none-any.whl.metadata
  Obtaining dependency information for threadpoolctl>=3.5.0 from https://files.pythonhosted.org/packages/32/d5/f9a850d79b0851d1d4ef6456097579a9005b31fea68726a4ae5f2d82ddd9/threadpoolctl-3.6.0-py3-none-any.whl.met


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: C:\Users\imama\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [1]:
pip install langgraph langchain langchain-openai pandas joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: C:\Users\imama\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import difflib
from dotenv import load_dotenv

load_dotenv()
pd.set_option('display.max_columns', None)

team = pd.read_csv('team_match_features_v1_2026-07-27.csv')
player = pd.read_csv('player_match_features_v1_2026-07-27.csv')

print("team rows:", len(team), "| player rows:", len(player))

team rows: 15808 | player rows: 274089


In [2]:
ARTIFACT_DIR = "models"

_match_model = joblib.load(f"{ARTIFACT_DIR}/match_winner_model.joblib")
_player_models ={
    "disposals": joblib.load(f"{ARTIFACT_DIR}/top_player_model_disposals.joblib"),
    "goals": joblib.load(f"{ARTIFACT_DIR}/top_player_model_goals.joblib"),
    "fantasy_points": joblib.load(f"{ARTIFACT_DIR}/top_player_model_fantasy_points.joblib"),
}
_latest_team_features = joblib.load(f"{ARTIFACT_DIR}/latest_team_features.joblib")
_latest_player_features = joblib.load(f"{ARTIFACT_DIR}/latest_player_features.joblib")
_valid_teams = joblib.load(f"{ARTIFACT_DIR}/valid_teams.joblib")

print(f"{len(_valid_teams)} valid teams loaded")
print(f"models loaded: match winner + 3 top player models (disposals, goals, fantasy_points)")

20 valid teams loaded
models loaded: match winner + 3 top player models (disposals, goals, fantasy_points)


In [3]:
MODEL = "smart"

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["API_KEY"],
)

## Task 1: Graph Design for the Full System

### State schema

The state needs to carry the user's question all the way through the graph, plus whatever each node adds along the way:

* `user_query`: the user's question.
* `conversation_history`: previous messages, so the agent remembers earlier parts of the conversation.
* `intent`: what the user is trying to do (intent classification) (`factual`, `retrieval`, `prediction`, or `off_topic`).
* `entities`: important information found in the question, like team names, player names, or the stat type.
* `tool_result`: the result returned by the retrieval or prediction tool.
* `error`: stores any error if something goes wrong (for example, if a team or player isn't found).
* `needs_clarification`: tells the graph if it should ask the user for more information instead of guessing.
* `clarification_question`: the question shown to the user when more information is needed.
* `final_response`: the final answer sent back to the user.
* `trace`: a log of what each node did, so it's easier to debug and see how the graph reached its answer.


### Graph sketch

```
                            user_query
                                 |
                                 |
                                 |
                                 ↓
                        +----------------+
                        |   router node   |
                        | (intent + entity|
                        |  extraction)    |
                        +--------+-------+
                                 |
           +---------------------+----------------------+---------------------+
           |                     |                       |                    |
     intent=retrieval     intent=prediction       intent=factual       intent=off_topic
           |                     |                       |                    |
   +-------v------+     +--------v-------+                |                    |
   | retrieval    |     | prediction     |                |                    |
   | node (stats, |     | node (match /  |                |                    |
   | head-to-head)|     | top player)    |                |                    |
   +-------+------+     +--------+-------+                |                    |
           |                     |                        |                    |
           +----------+----------+                        |                    |
                      |                                    |                    |
              +-------v--------+                           |                    |
              | validation node |                          |                    |
              | (did the tool   |                          |                    |
              |  actually work?)|                          |                    |
              +-------+--------+                           |                    |
                      |                                    |                    |
           needs_clarification? ---yes--> (ask user)        |                    |
                      | no                                  |                    |
                      +-----------------+--------------------+--------------------+
                                        |
                              +---------v----------+
                              | response formatting |
                              | node (adds          |
                              | probability wording |
                              | for predictions)     |
                              +---------+-----------+
                                        |
                                     final_response
```


| **Component**                | **Explanation**                                                                          |
| ---------------------------- | ---------------------------------------------------------------------------------------- |
| **user_query**               | The user's question enters the graph here.                                               |
| **router node**              | Figures out what the user wants and extracts things like team or player names.           |
| **intent = retrieval**       | Used when the user is asking for existing AFL stats or records.                          |
| **retrieval node**           | Looks up the requested information from the dataset.                                     |
| **intent = prediction**      | Used when the user is asking the model to make a prediction.                             |
| **prediction node**          | Calls the prediction models to generate match or player predictions.                     |
| **intent = factual**         | Used for general AFL questions that don't need a tool.                                   |
| **intent = off_topic**       | Used when the question isn't related to AFL.                                             |
| **validation node**          | Checks whether the retrieval or prediction tool returned a valid result.                 |
| **needs_clarification?**     | If something is missing or unclear, the graph asks the user instead of guessing.         |
| **response formatting node** | Formats the final answer and adds probabilities/disclaimers for predictions when needed. |
| **final_response**           | The final answer that gets sent back to the user.                                        |

### Justification: Why use explicit routing instead of one free agent?

A single agent decides on its own what to do, every time the user asks something. It chooses which tool to use and how to answer. While this is flexible, it also means it might forget important rules.

For example, prediction answers are supposed to sound like probabilities instead of certainties, but the agent could accidentally ignore that and make a prediction sound guaranteed.

Using LangGraph makes the process much more controlled. Every prediction always follows the same path: 

**prediction_node --> validation_node --> response_formatting_node**. 

This means every prediction is checked first, then the response is automatically formatted with a probability and a disclaimer before it's shown to the user. 

Retrieval results also go through validation, so if a team or player can't be found, the graph asks the user for clarification instead of guessing. This makes the system more reliable, consistent, and much easier to debug